<a href="https://colab.research.google.com/github/BarGinger/HCML-NLP-Project/blob/Bar/src/proto_lm/drugs_reviews_proto_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Using Proto-ML [1] to analyse the drug reviews dataset [2]

[1] https://github.com/yx131/proto-lm/tree/main

[2] https://www.kaggle.com/datasets/mohamedabdelwahabali/drugreview/data

### Global imports

In [ ]:
import argparse
!pip install pytorch-lightning
!pip install -U datasets
import torch
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification
from ProtoLM import proto_lm
from proto_data_class import sst_datamodule
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers.tensorboard import TensorBoardLogger
import datasets
from tqdm import tqdm
import numpy as np

### Define wrapper class for Proto-lm to get 1-10 values and not 0-9 values and improve training

In [ ]:
import torch.nn.functional as F
from collections import Counter

class ImprovedProtoLM(proto_lm):
    """
    ULTRA-CONSERVATIVE ProtoLM to prevent complete model collapse.
    Minimizes prototype learning and focuses on classification.
    """
    def __init__(self, class_weights=None, label_smoothing=0.1, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights
        self.label_smoothing = label_smoothing
        
        if self.class_weights is not None:
            print(f"Using class balancing with weights: {self.class_weights}")
        if self.label_smoothing > 0:
            print(f"Using label smoothing: {self.label_smoothing}")
    
    def calc_loss(self, logits, labels, similarities):
        """
        EMERGENCY OVERRIDE: Focus 95% on classification, 5% on prototypes
        """
        # 1. PRIMARY LOSS: Pure classification with heavy regularization
        if self.class_weights is not None:
            weights = self.class_weights.to(logits.device)
            classification_loss = F.cross_entropy(
                logits, labels, 
                weight=weights, 
                label_smoothing=self.label_smoothing
            )
        else:
            classification_loss = F.cross_entropy(
                logits, labels, 
                label_smoothing=self.label_smoothing
            )
        
        # 2. MINIMAL PROTOTYPE LOSS (if absolutely necessary)
        prototype_loss = torch.tensor(0.0, device=logits.device)
        
        # Only add prototype loss if lambda0 > 0 and very carefully
        if hasattr(self.hparams, 'lambda0') and self.hparams.lambda0 > 0:
            try:
                # Super minimal prototype regularization
                prototype_norms = torch.norm(self.prototypes, dim=1)
                prototype_loss = self.hparams.lambda0 * 0.01 * prototype_norms.mean()  # 100x reduction
            except Exception as e:
                print(f"Warning: Prototype loss failed, using zero: {e}")
                prototype_loss = torch.tensor(0.0, device=logits.device)
        
        # 3. TOTAL LOSS (classification dominates completely)
        total_loss = classification_loss + prototype_loss
        
        return {
            'classification_loss': classification_loss,
            'prototype_loss': prototype_loss,
            'total_loss': total_loss
        }
    
    def training_step(self, batch, batch_idx):
        """Ultra-conservative training step"""
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"] 
        sentiment_features = batch["sentiment_features"]
        labels = batch["labels"]
        
        # Forward pass
        outputs = self(
            input_ids=input_ids,
            attention_mask=attention_mask,
            sentiment_features=sentiment_features,
            labels=labels
        )
        
        # Use ultra-conservative loss
        try:
            if 'similarities' in outputs:
                loss_dict = self.calc_loss(outputs['probs'], labels, outputs['similarities'])
            else:
                # Pure classification fallback
                if self.class_weights is not None:
                    weights = self.class_weights.to(outputs['probs'].device)
                    loss = F.cross_entropy(outputs['probs'], labels, weight=weights, label_smoothing=self.label_smoothing)
                else:
                    loss = F.cross_entropy(outputs['probs'], labels, label_smoothing=self.label_smoothing)
                loss_dict = {'total_loss': loss, 'classification_loss': loss, 'prototype_loss': torch.tensor(0.0)}
        except Exception as e:
            print(f"Warning: Loss calculation failed, using basic cross-entropy: {e}")
            loss = F.cross_entropy(outputs['probs'], labels, label_smoothing=self.label_smoothing)
            loss_dict = {'total_loss': loss, 'classification_loss': loss, 'prototype_loss': torch.tensor(0.0)}
            
        loss = loss_dict['total_loss']
        
        # Log metrics
        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=True)
        self.log('train_cls_loss', loss_dict['classification_loss'], on_step=True, on_epoch=True)
        if 'prototype_loss' in loss_dict:
            self.log('train_proto_loss', loss_dict['prototype_loss'], on_step=True, on_epoch=True)
        
        # Calculate accuracy
        preds = torch.argmax(outputs['probs'], dim=1)
        accuracy = (preds == labels).float().mean()
        self.log('train_accuracy', accuracy, prog_bar=True, on_step=True, on_epoch=True)
        
        return loss
    
    def validation_step(self, batch, batch_idx):
        """Ultra-conservative validation step"""
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        sentiment_features = batch["sentiment_features"] 
        labels = batch["labels"]
        
        outputs = self(
            input_ids=input_ids,
            attention_mask=attention_mask,
            sentiment_features=sentiment_features,
            labels=labels
        )
        
        # Use conservative loss
        try:
            if 'similarities' in outputs:
                loss_dict = self.calc_loss(outputs['probs'], labels, outputs['similarities'])
            else:
                if self.class_weights is not None:
                    weights = self.class_weights.to(outputs['probs'].device)
                    loss = F.cross_entropy(outputs['probs'], labels, weight=weights, label_smoothing=self.label_smoothing)
                else:
                    loss = F.cross_entropy(outputs['probs'], labels, label_smoothing=self.label_smoothing)
                loss_dict = {'total_loss': loss}
        except Exception as e:
            print(f"Warning: Validation loss calculation failed: {e}")
            loss = F.cross_entropy(outputs['probs'], labels, label_smoothing=self.label_smoothing)
            loss_dict = {'total_loss': loss}
            
        loss = loss_dict['total_loss']
        
        # Calculate accuracy
        preds = torch.argmax(outputs['probs'], dim=1)
        accuracy = (preds == labels).float().mean()
        
        self.log('val_loss', loss, prog_bar=True, on_epoch=True)
        self.log('val_accuracy', accuracy, prog_bar=True, on_epoch=True)
        
        return {'val_loss': loss, 'val_accuracy': accuracy}

print("ULTRA-CONSERVATIVE ImprovedProtoLM class defined!")
print("EMERGENCY MODE: Minimal prototype learning, maximum classification focus")
print("Key safety features:")
print("   - 100x reduced prototype pressure")
print("   - Robust error handling") 
print("   - Pure classification fallback")
print("   - Heavy label smoothing regularization")

In [ ]:
from transformers import AutoConfig

# Parameters for Proto-LM training on the drug review dataset
model_name = 'bert-base-uncased'  # Backbone LLM model to load
# Maximum sentence length to pad/truncate to
model_name = 'bert-base-uncased'  # Backbone LLM model to load

# Function to calculate class weights for balanced training
def get_class_weights(drug_review_dm):
    """Calculate class weights to handle imbalance"""
    train_labels = []
    for batch in drug_review_dm.train_dataloader():
        train_labels.extend(batch["labels"].cpu().numpy())

    # Count frequency of each class
    class_counts = Counter(train_labels)
    print("Class distribution in training data:")
    for i in range(10):
        count = class_counts.get(i, 0)
        print(f"  Class {i} (Rating {i+1}): {count:,} samples")

    # Calculate inverse frequency weights
    total_samples = len(train_labels)
    class_weights = []
    for i in range(10):
        count = class_counts.get(i, 1)  # Avoid division by zero
        weight = total_samples / (10.0 * count)  # Inverse frequency
        class_weights.append(weight)

    # Normalize weights
    class_weights = torch.tensor(class_weights, dtype=torch.float32)
    class_weights = class_weights / class_weights.sum() * 10.0  # Normalize to sum to num_classes

    print(f"\nClass weights: {class_weights}")
    return class_weights

# ULTRA-CONSERVATIVE CONFIGURATION - Prevent complete collapse to single class
args = {
    'model_name': model_name,
    'max_seq_length': 100,
    'num_prototypes': 100,      # ✅ DRASTICALLY REDUCED from 500 (minimal complexity)
    'hidden_shape': 1024,       # Will be set dynamically
    'num_classes': 10,          # ✅ 10-class classification (not regression!)
    'cohsep_ratio': 0.01,       # ✅ ALMOST ZERO separation pressure (was 0.2)
    'lambda0': 0.001,           # ✅ MINIMAL prototype pressure (was 0.1)
    'lr': 1e-5,                 # ✅ VERY LOW learning rate (was 1e-4)
    'proto_training_weights': 1,
    'batch_size': 128,          # ✅ BACK TO 128 for stability (64 caused issues)
    'logger_dir': 'tb_logs_ultraconservative',
    'checkpoint_dir': 'ckpt_dir_ultraconservative',
    'config_subdir': 'ultraconservative_config',
    'max_epochs': 20,           # ✅ More epochs since learning rate is very low
    'num_gpu': 1,               # ✅ Fixed from 4 (single GPU)
    'load_model': model_name,
    'label_smoothing': 0.3,     # ✅ HIGH regularization (was 0.1)
}

# Dynamically fetch hidden size from the model configurationa
config = AutoConfig.from_pretrained(args['model_name'])
hidden_size = config.hidden_size  # 768 for bert-base-uncased
args['hidden_shape'] = hidden_size

print("ULTRA-CONSERVATIVE CONFIGURATION TO PREVENT COMPLETE COLLAPSE:")
print("EXTREME changes made to fix single-class prediction:")
print(f"  EMERGENCY: batch_size back to 128 (64 was too unstable)")
print(f"  EMERGENCY: lambda0 = {args['lambda0']} (was 0.1) - ALMOST NO prototype pressure")
print(f"  EMERGENCY: cohsep_ratio = {args['cohsep_ratio']} (was 0.2) - VIRTUALLY NO separation")
print(f"  EMERGENCY: learning rate = {args['lr']} (was 1e-4) - CRAWLING SPEED")
print(f"  EMERGENCY: prototypes = {args['num_prototypes']} (was 500) - DRASTIC REDUCTION")
print(f"  EMERGENCY: label smoothing = {args['label_smoothing']} (was 0.1) - HIGH REGULARIZATION")
print(f"  EMERGENCY: max_epochs = {args['max_epochs']} (compensating for slow learning)")

print(f'\n📋 Full configuration: {args}')

# Get data module
drug_review_dm = sst_datamodule(
    model_name_or_path=args['model_name'],
    max_seq_length=args['max_seq_length'],
    train_batch_size=args['batch_size'],
    eval_batch_size=args['batch_size']
)
drug_review_dm.setup(stage='fit')

# Calculate class weights after data is loaded
print("Calculating class weights for balanced training...")
class_weights = get_class_weights(drug_review_dm)
args['class_weights'] = class_weights

print(f"\nLoading base model: {args['load_model']}")

base_model = AutoModelForSequenceClassification.from_pretrained(
    args['model_name'],
    num_labels=args['num_classes'],  # ✅ Ensure correct number of output classes
    ignore_mismatched_sizes=True
)

if hasattr(base_model, "roberta"):
    llm_model = base_model.roberta
elif hasattr(base_model, "bert"):
    llm_model = base_model.bert
else:
    llm_model = base_model

# Create the IMPROVED proto model with class balancing and label smoothing
proto = ImprovedProtoLM(
    pretrained_model=llm_model,
    max_seq_length=args['max_seq_length'],
    num_prototypes=args['num_prototypes'],
    hidden_shape=args['hidden_shape'],
    num_classes=args['num_classes'],
    cohsep_ratio=args['cohsep_ratio'],
    lambda0=args['lambda0'],
    lr=args['lr'],
    proto_training_weights=bool(args['proto_training_weights']),
    class_weights=args['class_weights'],  # ✅ Add class balancing
    label_smoothing=args['label_smoothing']  # ✅ Add label smoothing
)

print("Model and data setup complete with ULTRA-CONSERVATIVE configuration!")
print("EMERGENCY MODE: Preventing single-class collapse")
print("Using ImprovedProtoLM with minimal prototype pressure.")
print("If this fails, fallback SimpleBERTClassifier is available!")
print("\nKEY CHANGES:")
print(f"   Batch size: 128 (stable)")
print(f"   Learning rate: {args['lr']} (ultra-slow)")
print(f"   Lambda0: {args['lambda0']} (almost zero prototype pressure)")
print(f"   Prototypes: {args['num_prototypes']} (minimal)")
print(f"   Label smoothing: {args['label_smoothing']} (high regularization)")

## Model Collapse Prevention Strategy

**Problem:** Previous models collapsed to predicting only a single rating class.

**Root Causes:**
1. **Small batch size** - Caused training instability and poor gradient estimates
2. **Aggressive prototype learning** - Forced model into local minima
3. **Impatient early stopping** - Model stopped before proper learning

**Conservative Solution:**
- `lambda0 = 0.001` (minimal prototype pressure)
- `cohsep_ratio = 0.01` (almost no separation pressure) 
- `learning_rate = 1e-5` (very slow, stable learning)
- `batch_size = 128` (back to stable size)
- `num_prototypes = 100` (reduced complexity)
- `label_smoothing = 0.3` (high regularization)
- `patience = 15` (much more patient)
- `full precision` (no mixed precision instability)

**Alternative Approaches:**
1. **Pure Classification**: Disable prototype learning entirely (`lambda0=0`)
2. **Gradual Introduction**: Start with `lambda0=0`, then gradually increase
3. **Different Architecture**: Try standard BERT classifier first, then add prototypes
4. **Learning Rate Scheduling**: Start even lower, then increase gradually

In [ ]:
# 🔬 DIAGNOSTIC: Check the ultra-conservative configuration
print("ULTRA-CONSERVATIVE CONFIGURATION SUMMARY:")
print("=" * 60)
print(f"Batch Size: {args['batch_size']} (FIXED: was 64, caused instability)")
print(f"Learning Rate: {args['lr']} (100x slower than original)")
print(f"Lambda0: {args['lambda0']} (1000x weaker than original)")
print(f"Cohesion/Separation: {args['cohsep_ratio']} (50x weaker than original)")
print(f"Num Prototypes: {args['num_prototypes']} (10x fewer than original)")
print(f"Label Smoothing: {args['label_smoothing']} (3x stronger regularization)")
print(f"Max Epochs: {args['max_epochs']} (compensating for slow learning)")
print(f"Early Stop Patience: 15 (3x more patient)")
print("=" * 60)

print("\nEXPECTED BEHAVIOR:")
print("   Much slower training (due to tiny learning rate)")
print("   More stable gradients (due to larger batch size)")
print("   Less prototype interference (due to tiny lambda0)")
print("   Better regularization (due to high label smoothing)")
print("   More diverse predictions (due to minimal prototype pressure)")

print("\nIF STILL FAILS:")
print("   The issue might be fundamental to ProtoLM architecture")
print("   → Use SimpleBERTClassifier fallback (pure BERT)")
print("   → Or completely disable prototypes (lambda0=0)")

# Quick sanity check on class weights
if 'class_weights' in args and args['class_weights'] is not None:
    print(f"\nClass weights (for balancing): {args['class_weights']}")
    min_weight = torch.min(args['class_weights']).item()
    max_weight = torch.max(args['class_weights']).item()
    print(f"   Weight range: {min_weight:.3f} to {max_weight:.3f} (ratio: {max_weight/min_weight:.1f}x)")
else:
    print("\nNo class weights applied")

## Improved ProtoLM Implementation

In [ ]:
class RatingPredictionWrapper:
    """
    Wrapper to convert ProtoLM 0-9 class predictions back to 1-10 drug ratings.
    Handles both single predictions and batches.
    """
    def __init__(self, model):
        self.model = model
        self.model.eval()

    def predict_rating(self, input_ids, attention_mask, sentiment_features):
        """
        Predict drug rating (1-10) from model output.

        Args:
            input_ids: tokenized text
            attention_mask: attention mask
            sentiment_features: sentiment scores

        Returns:
            ratings: 1-10 rating predictions (numpy array)
            probabilities: class probabilities (numpy array)
            confidence: prediction confidence (numpy array)
        """
        with torch.no_grad():
            outputs = self.model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                sentiment_features=sentiment_features
            )

            # Get class probabilities and predictions
            probs = torch.softmax(outputs['probs'], dim=1)  # Convert logits to probabilities
            class_preds = torch.argmax(probs, dim=1)  # 0-9 class indices

            # Convert 0-9 classes to 1-10 ratings
            ratings = class_preds + 1  # Simple mapping: class 0 -> rating 1, ..., class 9 -> rating 10

            # Calculate confidence (max probability)
            confidence = torch.max(probs, dim=1)[0]

            return (
                ratings.cpu().numpy(),
                probs.cpu().numpy(),
                confidence.cpu().numpy()
            )

    def predict_single(self, text, tokenizer, device='cpu'):
        """
        Predict rating for a single text review.

        Args:
            text: raw review text
            tokenizer: tokenizer to use
            device: device to run on

        Returns:
            rating: predicted rating (1-10)
            confidence: prediction confidence (0-1)
        """
        # Tokenize
        encoded = tokenizer(
            text,
            truncation=True,
            padding=True,
            max_length=100,  # Match training max_length
            return_tensors='pt'
        )

        # Dummy sentiment features (zeros) - in practice you'd compute these
        batch_size = encoded['input_ids'].shape[0]
        sentiment_features = torch.zeros(batch_size, 4)  # [neg, neu, pos, compound]

        # Move to device
        encoded = {k: v.to(device) for k, v in encoded.items()}
        sentiment_features = sentiment_features.to(device)

        # Predict
        ratings, probs, confidence = self.predict_rating(
            encoded['input_ids'],
            encoded['attention_mask'],
            sentiment_features
        )

        return ratings[0], confidence[0]

print("✅ RatingPredictionWrapper defined for converting model outputs to 1-10 ratings!")

In [ ]:
def evaluate_model_comprehensive(model, data_module, device='cpu'):
    """
    Comprehensive evaluation with binary collapse detection and diversity analysis.
    """
    from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                                confusion_matrix, classification_report,
                                mean_squared_error, mean_absolute_error)
    import matplotlib.pyplot as plt
    import seaborn as sns

    print("🔄 Running comprehensive model evaluation...")

    # Create prediction wrapper
    wrapper = RatingPredictionWrapper(model)

    # Get test dataloader
    test_dataloader = data_module.test_dataloader()

    all_ratings = []  # 1-10 ratings
    all_true_ratings = []  # 1-10 true ratings
    all_class_preds = []  # 0-9 class predictions
    all_true_classes = []  # 0-9 true classes
    all_confidences = []

    print("🔄 Processing test batches...")
    for batch in tqdm(test_dataloader, desc="Evaluating"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        sentiment_features = batch["sentiment_features"].to(device)
        true_classes = batch["labels"].cpu().numpy()

        # Get predictions
        ratings, probs, confidences = wrapper.predict_rating(
            input_ids, attention_mask, sentiment_features
        )

        class_preds = ratings - 1  # Convert back to 0-9 for analysis
        true_ratings = true_classes + 1  # Convert to 1-10

        all_ratings.extend(ratings)
        all_true_ratings.extend(true_ratings)
        all_class_preds.extend(class_preds)
        all_true_classes.extend(true_classes)
        all_confidences.extend(confidences)

    # Convert to numpy arrays
    all_ratings = np.array(all_ratings)
    all_true_ratings = np.array(all_true_ratings)
    all_class_preds = np.array(all_class_preds)
    all_true_classes = np.array(all_true_classes)
    all_confidences = np.array(all_confidences)

    print(f"✅ Processed {len(all_ratings)} test samples")

    # 1. CHECK FOR BINARY COLLAPSE
    unique_preds = np.unique(all_class_preds)
    print(f"\n🔍 BINARY COLLAPSE CHECK:")
    print(f"   Unique predicted classes: {unique_preds}")
    print(f"   Number of unique predictions: {len(unique_preds)}")

    if len(unique_preds) <= 2:
        print("   ⚠️  WARNING: Model appears to have collapsed to binary predictions!")
    elif len(unique_preds) <= 5:
        print("   ⚠️  WARNING: Model has limited prediction diversity")
    else:
        print("   ✅ Good prediction diversity")

    # 2. PREDICTION DISTRIBUTION ANALYSIS
    print(f"\n📊 PREDICTION DISTRIBUTION:")
    unique_ratings, counts = np.unique(all_ratings, return_counts=True)
    for rating, count in zip(unique_ratings, counts):
        percentage = count / len(all_ratings) * 100
        print(f"   Rating {rating}: {count:,} samples ({percentage:.1f}%)")

    # 3. CLASSIFICATION METRICS (0-9 classes)
    accuracy = accuracy_score(all_true_classes, all_class_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_true_classes, all_class_preds, average='weighted', zero_division=0)
    macro_f1 = precision_recall_fscore_support(all_true_classes, all_class_preds, average='macro', zero_division=0)[2]

    print(f"\n📈 CLASSIFICATION METRICS (Classes 0-9):")
    print(f"   Accuracy: {accuracy:.4f}")
    print(f"   Weighted F1: {f1:.4f}")
    print(f"   Macro F1: {macro_f1:.4f}")
    print(f"   Weighted Precision: {precision:.4f}")
    print(f"   Weighted Recall: {recall:.4f}")

    # 4. RATING REGRESSION METRICS (1-10 ratings)
    mse = mean_squared_error(all_true_ratings, all_ratings)
    mae = mean_absolute_error(all_true_ratings, all_ratings)
    rmse = np.sqrt(mse)

    print(f"\n📊 RATING REGRESSION METRICS (Ratings 1-10):")
    print(f"   MSE: {mse:.4f}")
    print(f"   MAE: {mae:.4f}")
    print(f"   RMSE: {rmse:.4f}")

    # 5. CONFIDENCE ANALYSIS
    avg_confidence = np.mean(all_confidences)
    print(f"\n🎯 CONFIDENCE ANALYSIS:")
    print(f"   Average confidence: {avg_confidence:.4f}")
    print(f"   Min confidence: {np.min(all_confidences):.4f}")
    print(f"   Max confidence: {np.max(all_confidences):.4f}")

    # 6. CONFUSION MATRIX VISUALIZATION
    print(f"\n📋 Generating confusion matrix...")
    plt.figure(figsize=(12, 10))
    conf_matrix = confusion_matrix(all_true_ratings, all_ratings, labels=range(1, 11))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=[f'Rating {i}' for i in range(1, 11)],
                yticklabels=[f'Rating {i}' for i in range(1, 11)])
    plt.title('Confusion Matrix - Drug Rating Predictions (1-10)')
    plt.xlabel('Predicted Rating')
    plt.ylabel('True Rating')
    plt.tight_layout()
    plt.show()

    # 7. RETURN COMPREHENSIVE RESULTS
    return {
        'accuracy': accuracy,
        'weighted_f1': f1,
        'macro_f1': macro_f1,
        'mse': mse,
        'mae': mae,
        'rmse': rmse,
        'avg_confidence': avg_confidence,
        'unique_predictions': len(unique_preds),
        'binary_collapse': len(unique_preds) <= 2,
        'predictions': all_ratings,
        'true_ratings': all_true_ratings,
        'confidences': all_confidences
    }

print("✅ Improved evaluation function defined with binary collapse detection!")

In [ ]:
# get training utilities like logger and checkpoints with IMPROVED SETUP
from pytorch_lightning.loggers.tensorboard import TensorBoardLogger
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor

# Enhanced logger
tb_logger = TensorBoardLogger(f"{args['logger_dir']}", name="improved_drug_review_logs")

# Enhanced checkpoint callback - monitor accuracy instead of just loss
ckpt_path = f"{args['checkpoint_dir']}/{args['config_subdir']}"
checkpoint_callback = ModelCheckpoint(
    dirpath=ckpt_path,
    monitor='val_accuracy',  # ✅ Monitor accuracy for better model selection
    save_top_k=3,
    mode='max',  # ✅ Maximize accuracy
    filename="{epoch}-{val_accuracy:.4f}-{val_loss:.4f}",
    save_last=True,
    verbose=True
)

# Enhanced early stopping - MUCH MORE PATIENT for ultra-slow learning
early_stop_callback = EarlyStopping(
    monitor='val_accuracy',  # ✅ Monitor accuracy
    min_delta=0.0001,        # ✅ TINY minimum improvement (was 0.001)
    patience=15,             # ✅ VERY PATIENT (was 5) - slow learning needs time
    verbose=True,
    mode='max'  # ✅ Maximize accuracy
)

# Learning rate monitoring
lr_monitor = LearningRateMonitor(logging_interval='epoch')

# ULTRA-CONSERVATIVE trainer settings
trainer = pl.Trainer(
    max_epochs=args['max_epochs'],
    accelerator="auto",
    devices=1,
    logger=tb_logger,
    callbacks=[checkpoint_callback, early_stop_callback, lr_monitor],
    gradient_clip_val=0.1,   # ✅ VERY GENTLE gradient clipping (was 1.0)
    precision=32,            # ✅ FULL precision for maximum stability
    deterministic=True,      # ✅ Reproducible results
    enable_checkpointing=True,
    log_every_n_steps=200,   # ✅ Less frequent logging for stability
    val_check_interval=1.0,  # ✅ Validate once per epoch
    accumulate_grad_batches=1,
    enable_progress_bar=True,
    detect_anomaly=False     # ✅ Disable for speed (we have error handling)
)

print("🚀 Starting improved training with enhanced monitoring and stability...")
trainer.fit(proto, datamodule=drug_review_dm)

# Optionally test or save misclassified
# trainer.test(proto, datamodule=drug_review_dm, ckpt_path=args['load_model'])
# torch.save(proto.misclassified, 'drug_review_logs/misclassed.pt')

## Clean ProtoLM Implementation

ProtoLM implementation following the tutorial pattern for drug review classification.

In [ ]:
class DrugReviewProtoLM(proto_lm):
    """ProtoLM adapted for drug review classification following the tutorial pattern."""
    
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        print(f"DrugReviewProtoLM initialized:")
        print(f"  Prototypes: {self.hparams.num_prototypes}")
        print(f"  Classes: {self.hparams.num_classes}")
        print(f"  Lambda0: {self.hparams.lambda0}")
        print(f"  Cohsep ratio: {self.hparams.cohsep_ratio}")
    
    def training_step(self, batch, batch_idx):
        """Training step following the tutorial pattern"""
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]
        
        # Handle sentiment features
        if "sentiment_features" in batch:
            sentiment_features = batch["sentiment_features"]
        else:
            batch_size = input_ids.shape[0]
            sentiment_features = torch.zeros(batch_size, 4, device=input_ids.device)
        
        # Forward pass
        outputs = self(
            input_ids=input_ids,
            attention_mask=attention_mask,
            sentiment_features=sentiment_features,
            labels=labels
        )
        
        # Calculate loss
        similarities = outputs['similarities']
        logits = outputs['logits']
        all_losses = self.calc_loss(logits, labels, similarities)
        total_loss = all_losses['total_loss']
        
        # Calculate accuracy
        preds = torch.argmax(outputs['probs'], dim=1)
        accuracy = (preds == labels).float().mean()
        
        # Log metrics
        self.log('train_loss', total_loss, prog_bar=True, on_step=True, on_epoch=True)
        self.log('train_accuracy', accuracy, prog_bar=True, on_step=True, on_epoch=True)
        self.log('train_ce_loss', all_losses['ce_loss'], on_step=False, on_epoch=True)
        self.log('train_cohesion_loss', all_losses['cohesion_loss'], on_step=False, on_epoch=True)
        self.log('train_separation_loss', all_losses['separation_loss'], on_step=False, on_epoch=True)
        
        return total_loss
    
    def validation_step(self, batch, batch_idx):
        """Validation step following the tutorial pattern"""
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]
        
        # Handle sentiment features
        if "sentiment_features" in batch:
            sentiment_features = batch["sentiment_features"]
        else:
            batch_size = input_ids.shape[0]
            sentiment_features = torch.zeros(batch_size, 4, device=input_ids.device)
        
        # Forward pass
        outputs = self(
            input_ids=input_ids,
            attention_mask=attention_mask,
            sentiment_features=sentiment_features,
            labels=labels
        )
        
        # Calculate loss
        similarities = outputs['similarities']
        logits = outputs['logits']
        all_losses = self.calc_loss(logits, labels, similarities)
        val_loss = all_losses['total_loss']
        
        # Calculate accuracy
        preds = torch.argmax(outputs['probs'], dim=1)
        accuracy = (preds == labels).float().mean()
        
        # Log metrics
        self.log('val_loss', val_loss, prog_bar=True, on_epoch=True)
        self.log('val_accuracy', accuracy, prog_bar=True, on_epoch=True)
        
        return {'val_loss': val_loss, 'val_accuracy': accuracy}

# Create the model with tutorial-based configuration
tutorial_args = {
    'model_name': model_name,
    'max_seq_length': 100,
    'num_prototypes': 200,     # Tutorial uses 200
    'hidden_shape': 768,       # BERT base hidden size
    'num_classes': 10,
    'cohsep_ratio': 0.5,       # Tutorial default
    'lambda0': 0.1,            # Tutorial default
    'lr': 1e-4,                # Standard learning rate
    'proto_training_weights': 1,
    'batch_size': 128,
    'max_epochs': 10,
}

print("Configuration (following tutorial):")
print(f"  Prototypes: {tutorial_args['num_prototypes']}")
print(f"  Lambda0: {tutorial_args['lambda0']}")
print(f"  Cohsep ratio: {tutorial_args['cohsep_ratio']}")
print(f"  Learning rate: {tutorial_args['lr']}")
print(f"  Batch size: {tutorial_args['batch_size']}")

# Create the model
corrected_proto = DrugReviewProtoLM(
    pretrained_model=llm_model,
    max_seq_length=tutorial_args['max_seq_length'],
    num_prototypes=tutorial_args['num_prototypes'],
    hidden_shape=tutorial_args['hidden_shape'],
    num_classes=tutorial_args['num_classes'],
    cohsep_ratio=tutorial_args['cohsep_ratio'],
    lambda0=tutorial_args['lambda0'],
    lr=tutorial_args['lr'],
    proto_training_weights=bool(tutorial_args['proto_training_weights'])
)

print("ProtoLM model created successfully")

In [ ]:
# Training setup following tutorial pattern
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers.tensorboard import TensorBoardLogger

print("Setting up trainer following tutorial pattern...")

# Logger
corrected_logger = TensorBoardLogger('tb_logs_corrected', name='drug_review_logs')

# Checkpoint callback
corrected_checkpoint = ModelCheckpoint(
    dirpath='ckpt_dir_corrected/config',
    monitor='val_loss',
    save_top_k=3,
    filename="{epoch}-{val_loss:.4f}-{val_accuracy:.4f}",
    save_last=True,
    verbose=True
)

# Trainer
corrected_trainer = pl.Trainer(
    max_epochs=tutorial_args['max_epochs'],
    accelerator="auto",
    devices=1,
    logger=corrected_logger,
    callbacks=[corrected_checkpoint],
    precision=32,
    enable_progress_bar=True,
    log_every_n_steps=50,
    val_check_interval=1.0
)

print("Trainer setup complete")

In [ ]:
# Train the ProtoLM model
print("Starting ProtoLM training...")

try:
    corrected_trainer.fit(corrected_proto, datamodule=drug_review_dm)
    
    print("Training completed!")
    
    # Get final metrics
    if hasattr(corrected_trainer, 'callback_metrics'):
        final_val_acc = corrected_trainer.callback_metrics.get('val_accuracy', 0)
        final_val_loss = corrected_trainer.callback_metrics.get('val_loss', float('inf'))
        print(f"Final validation accuracy: {final_val_acc:.4f}")
        print(f"Final validation loss: {final_val_loss:.4f}")
        
        if final_val_acc > 0.15:
            print("Training successful - model is learning!")
        else:
            print("Low accuracy - may need hyperparameter tuning")
    
except Exception as e:
    print(f"Training failed: {str(e)}")
    import traceback
    traceback.print_exc()
    print("Check data module compatibility, ProtoLM forward pass, loss calculation, or device placement")

In [ ]:
# Test ProtoLM forward pass
print("Testing ProtoLM forward pass...")

test_batch = next(iter(drug_review_dm.train_dataloader()))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
corrected_proto = corrected_proto.to(device)

try:
    corrected_proto.eval()
    
    with torch.no_grad():
        # Small test batch
        input_ids = test_batch['input_ids'][:4].to(device)
        attention_mask = test_batch['attention_mask'][:4].to(device)
        labels = test_batch['labels'][:4].to(device)
        
        if 'sentiment_features' in test_batch:
            sentiment_features = test_batch['sentiment_features'][:4].to(device)
        else:
            sentiment_features = torch.zeros(4, 4, device=device)
        
        print(f"Input shapes moved to {device}:")
        print(f"  input_ids: {input_ids.shape}")
        print(f"  attention_mask: {attention_mask.shape}")
        print(f"  sentiment_features: {sentiment_features.shape}")
        print(f"  labels: {labels.shape}")
        
        # Forward pass
        outputs = corrected_proto(
            input_ids=input_ids,
            attention_mask=attention_mask,
            sentiment_features=sentiment_features
        )
        
        print("Forward pass successful!")
        print(f"  Logits shape: {outputs['probs'].shape}")
        print(f"  Similarities shape: {outputs['similarities'].shape}")
        
        # Test loss calculation
        all_losses = corrected_proto.calc_loss(outputs['logits'], labels, outputs['similarities'])
        print("Loss calculation successful!")
        print(f"  CE loss: {all_losses['ce_loss'].item():.4f}")
        print(f"  Cohesion loss: {all_losses['cohesion_loss'].item():.4f}")
        print(f"  Separation loss: {all_losses['separation_loss'].item():.4f}")
        print(f"  Total loss: {all_losses['total_loss'].item():.4f}")
        
        # Check predictions
        preds = torch.argmax(outputs['probs'], dim=1)
        print("Predictions vs Labels:")
        print(f"  Predictions: {preds.cpu().tolist()}")
        print(f"  Labels: {labels.cpu().tolist()}")
        print(f"  Unique predictions: {torch.unique(preds).cpu().tolist()}")
        
        print("ProtoLM is ready for training!")
        
except Exception as e:
    print(f"Forward pass failed: {str(e)}")
    import traceback
    traceback.print_exc()
    print("This indicates an issue with data batch format, ProtoLM forward method, device placement, or tensor shapes")

corrected_proto.train()

### Save the trained model

In [ ]:
# Save the complete model for deployment and reproducibility
import os
import json
from datetime import datetime

# Create a timestamped save directory
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_dir = f"final_proto_model_{timestamp}"
os.makedirs(save_dir, exist_ok=True)

print(f"Saving complete model to: {save_dir}")

# 1. Save the PyTorch Lightning checkpoint (includes everything)
trainer.save_checkpoint(os.path.join(save_dir, "model_checkpoint.ckpt"))

# 2. Save just the model weights (for loading into other frameworks)
torch.save(proto.state_dict(), os.path.join(save_dir, "model_weights.pt"))

# 3. Save the tokenizer (essential for preprocessing)
tokenizer = drug_review_dm.tokenizer
tokenizer.save_pretrained(os.path.join(save_dir, "tokenizer"))

# 4. Save model configuration and training arguments
model_config = {
    "model_name": args['model_name'],
    "max_seq_length": args['max_seq_length'],
    "num_prototypes": args['num_prototypes'],
    "hidden_shape": args['hidden_shape'],
    "num_classes": args['num_classes'],
    "cohsep_ratio": args['cohsep_ratio'],
    "lambda0": args['lambda0'],
    "lr": args['lr'],
    "proto_training_weights": args['proto_training_weights'],
    "batch_size": args['batch_size'],
    "max_epochs": args['max_epochs'],
    "timestamp": timestamp,
    "model_type": "ProtoLM_regression"
}

with open(os.path.join(save_dir, "model_config.json"), "w") as f:
    json.dump(model_config, f, indent=2)

# 5. Save data preprocessing info
data_config = {
    "text_fields": drug_review_dm.text_fields,
    "num_labels": drug_review_dm.num_labels,
    "max_seq_length": drug_review_dm.max_seq_length,
    "loader_columns": drug_review_dm.loader_columns,
    "model_name_or_path": drug_review_dm.model_name_or_path
}

with open(os.path.join(save_dir, "data_config.json"), "w") as f:
    json.dump(data_config, f, indent=2)

# 6. Save training metrics/logs if available
try:
    if hasattr(trainer.logger, 'log_dir'):
        import shutil
        shutil.copytree(trainer.logger.log_dir, os.path.join(save_dir, "logs"))
except:
    print("Could not copy training logs")

print(f"✅ Model saved successfully!")
print(f"📁 Save directory: {save_dir}")
print(f"📋 Contents:")
print(f"   - model_checkpoint.ckpt (PyTorch Lightning checkpoint)")
print(f"   - model_weights.pt (PyTorch state dict)")
print(f"   - tokenizer/ (Hugging Face tokenizer)")
print(f"   - model_config.json (model configuration)")
print(f"   - data_config.json (data preprocessing config)")
print(f"   - logs/ (training logs, if available)")

In [ ]:
!zip -r /content/final_proto_model_20250630_212135.zip /content/final_proto_model_20250630_212135


### Calc Quantus Metrics

In [ ]:
# 🎯 IMPROVED EVALUATION WITH BINARY COLLAPSE DETECTION
print("🔄 Running comprehensive evaluation with binary collapse detection...")

# Run improved evaluation
eval_results = evaluate_model_comprehensive(proto, drug_review_dm, device=proto.device)

print(f"\n📊 FINAL EVALUATION SUMMARY:")
print(f"   Binary Collapse: {'❌ YES' if eval_results['binary_collapse'] else '✅ NO'}")
print(f"   Unique Predictions: {eval_results['unique_predictions']}/10 classes")
print(f"   Classification Accuracy: {eval_results['accuracy']:.4f}")
print(f"   Weighted F1-Score: {eval_results['weighted_f1']:.4f}")
print(f"   Rating RMSE: {eval_results['rmse']:.4f}")
print(f"   Average Confidence: {eval_results['avg_confidence']:.4f}")

if not eval_results['binary_collapse']:
    print("\n🎉 SUCCESS: Model is predicting across multiple classes!")
    print("✅ The ultra-conservative configuration prevented collapse.")
else:
    print("\n🚨 CRITICAL: Model still collapsed with ultra-conservative settings!")
    print("\n💡 EMERGENCY FALLBACK OPTIONS:")
    print("1. 🎯 TRY FALLBACK: Use SimpleBERTClassifier (no prototypes)")
    print("2. 🔄 DISABLE PROTOTYPES: Set lambda0=0.0 completely")
    print("3. 📉 EVEN SLOWER: Try lr=5e-6 or lr=1e-6")
    print("4. 🎭 MORE SMOOTHING: Try label_smoothing=0.5")
    print("\n📋 TO USE FALLBACK CLASSIFIER:")
    print("   # First, try this simple approach:")
    print("   fallback_model = SimpleBERTClassifier(args['model_name'], class_weights=class_weights)")
    print("   trainer_fallback = pl.Trainer(max_epochs=5, accelerator='auto', devices=1)")
    print("   trainer_fallback.fit(fallback_model, datamodule=drug_review_dm)")
    print("   eval_fallback = evaluate_model_comprehensive(fallback_model, drug_review_dm)")
    print("\n🎯 The fallback should work since it's pure BERT classification!")

# Additional analysis
import pandas as pd
pred_counts = pd.Series(eval_results['predictions']).value_counts().sort_index()
print(f"\n📈 Prediction distribution:")
for rating, count in pred_counts.items():
    print(f"   Rating {rating}: {count} samples")

In [ ]:
import pandas as pd
print(f"value counts in real labels: {pd.Series(all_labels).value_counts()}\n")
print(f"value counts in predicted labels: {pd.Series(all_predictions).value_counts()}")

In [ ]:
import quantus
import torch
import numpy as np
import pandas as pd

# Define your model and data
model = proto  # Your Proto-LM model
model_name_for_csv = f"ProtoLM_{model_name}"  # Add your model name here
model.eval()  # Set the model to evaluation mode

# Define a wrapper for your model to work with Quantus
class ModelWrapper:
    def __init__(self, model):
        self.model = model

    def __call__(self, input_ids, attention_mask):
        with torch.no_grad():
            outputs = self.model.LLM(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True
            )
            logits = outputs.logits  # Assuming logits are the output
        return logits

wrapped_model = ModelWrapper(model)

# Define a sample batch of data (input_ids and attention_mask)
batch = next(iter(drug_review_dm.test_dataloader()))
input_ids = batch["input_ids"]
attention_mask = batch["attention_mask"]
sentiment_features = torch.cat((
    batch["sentiment_neg"],
    batch["sentiment_neu"],
    batch["sentiment_pos"],
    batch["sentiment_compound"]
), dim=1)
labels = batch["labels"]

# Pass sentiment features to the model
outputs = proto(
    input_ids=input_ids,
    attention_mask=attention_mask,
    sentiment_features=sentiment_features,
    labels=labels
)

# Define an attribution method (e.g., Integrated Gradients)
from captum.attr import IntegratedGradients
ig = IntegratedGradients(wrapped_model)

# Generate attributions for the input
attributions = ig.attribute(inputs=input_ids, additional_forward_args=(attention_mask,), target=labels)

# Convert attributions to numpy for Quantus
attributions_np = attributions.detach().cpu().numpy()

# Define Quantus metrics
metrics = {
    "Sparsity": quantus.Sparsity(),
    "Complexity": quantus.Complexity(),
    "Faithfulness": quantus.FaithfulnessCorrelation(),
    "Robustness": quantus.LocalLipschitzEstimate(),
    "Sensitivity": quantus.SensitivityN()
}

# Evaluate metrics
results = {}
for metric_name, metric in metrics.items():
    result = metric(
        model=wrapped_model,
        x_batch=input_ids.cpu().numpy(),
        y_batch=labels.cpu().numpy(),
        a_batch=attributions_np,
        explain_func=lambda x: attributions_np  # Use precomputed attributions
    )
    results[metric_name] = result

# Print results
for metric_name, result in results.items():
    print(f"{metric_name}: {result}")

# Export results to CSV
results_df = pd.DataFrame.from_dict(results, orient="index", columns=["Score"])
results_df["Model"] = model_name_for_csv  # Add model name to the DataFrame
results_df.reset_index(inplace=True)
results_df.rename(columns={"index": "Metric"}, inplace=True)

# Save to CSV
csv_filename = f"quantus_metrics_{model_name_for_csv}.csv"
results_df.to_csv(csv_filename, index=False)
print(f"Results saved to {csv_filename}")

### Plot similarity between test set cases and the learned concepts similar to figure 3 in their paper

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F

# Get prototype vectors (concepts)
prototypes = proto.prototypes.detach().cpu().numpy()  # shape: (num_prototypes, hidden_dim)

# Get representations for some samples (e.g., from your test set)
batch = next(iter(drug_review_dm.test_dataloader()))
with torch.no_grad():
    # Get last hidden states for the batch
    llm_out = proto.LLM(
        input_ids=batch['input_ids'].to(proto.device),
        attention_mask=batch['attention_mask'].to(proto.device),
        output_hidden_states=True
    )
    sample_reps = llm_out.hidden_states[-1][:, 0, :].cpu().numpy()  # [CLS] token or mean pooling



# prototypes: (num_prototypes, hidden_dim)
# sample_reps: (batch_size, hidden_dim)
# Assume proto.prototype_class_vec exists and is (num_prototypes, num_classes)

# 1. Identify positive and negative prototypes
# If proto.prototype_class_vec is one-hot or softmax over classes:
proto_class = proto.prototype_class_vec.detach().cpu().numpy()  # (num_prototypes, num_classes)
positive_proto_idx = np.argmax(proto_class[:, 1])  # class 1 = positive
negative_proto_idx = np.argmax(proto_class[:, 0])  # class 0 = negative

positive_proto = torch.tensor(prototypes[positive_proto_idx])
negative_proto = torch.tensor(prototypes[negative_proto_idx])

# 2. Compute similarities for each sample
sample_vecs = torch.tensor(sample_reps)  # (batch_size, hidden_dim)
sim_pos = F.cosine_similarity(sample_vecs, positive_proto.unsqueeze(0), dim=1)
sim_neg = F.cosine_similarity(sample_vecs, negative_proto.unsqueeze(0), dim=1)

# 3. Get ground-truth labels for the batch
labels = batch['labels'].cpu().numpy()

# 4. Plot
plt.figure(figsize=(8, 8))
for label in np.unique(labels):
    idxs = np.where(labels == label)[0]
    plt.scatter(sim_pos[idxs], sim_neg[idxs], label=f"Class {label}", alpha=0.7)
plt.xlabel("Similarity to Positive Prototype")
plt.ylabel("Similarity to Negative Prototype")
plt.title("2D Prototypical Space (like Proto-LM Fig. 3)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import torch.nn.functional as F

sample_vec = sample_reps[0]  # pick one sample
proto_vecs = torch.tensor(prototypes)
similarities = F.cosine_similarity(torch.tensor(sample_vec).unsqueeze(0), proto_vecs)
plt.bar(range(len(similarities)), similarities.numpy())
plt.xlabel("Prototype Index")
plt.ylabel("Cosine Similarity")
plt.title("Sample-Prototype Similarity")
plt.show()

In [ ]:
import torch
import pandas as pd
import numpy as np
import torch.nn.functional as F

# 1. Get all review texts and their embeddings from the training set
train_dataset = drug_review_dm.dataset["train"]
all_texts = train_dataset["review"]

# Get all input_ids and attention_mask for the train set
input_ids = train_dataset["input_ids"]
attention_mask = train_dataset["attention_mask"]

# Compute all embeddings (CLS token)
all_embeddings = []
batch_size = 128
for i in range(0, len(input_ids), batch_size):
    batch_input_ids = input_ids[i:i+batch_size].to(proto.device)
    batch_attention_mask = attention_mask[i:i+batch_size].to(proto.device)
    with torch.no_grad():
        outputs = proto.LLM(
            input_ids=batch_input_ids,
            attention_mask=batch_attention_mask,
            output_hidden_states=True
        )
        batch_embeds = outputs.hidden_states[-1][:, 0, :].cpu()  # CLS token
        all_embeddings.append(batch_embeds)
all_embeddings = torch.cat(all_embeddings, dim=0)  # (num_samples, hidden_dim)

# 2. For each prototype, find the closest text
prototypes = proto.prototypes.detach().cpu()  # (num_prototypes, hidden_dim)
closest_texts = []
for proto_vec in prototypes:
    sims = F.cosine_similarity(all_embeddings, proto_vec.unsqueeze(0), dim=1)
    idx = torch.argmax(sims).item()
    closest_texts.append(all_texts[idx])

# 3. Save to CSV
df = pd.DataFrame({"prototype_index": range(len(closest_texts)), "closest_text": closest_texts})
df.to_csv("prototype_texts.csv", index=False)